In [7]:
import pandas as pd
import numpy as np

In [4]:
# Variables
prices_df = []
trades_df = []
days=(-2, -1, 0)
base_dir="datasets-2024"

In [ ]:
# Load Price
for d in days:
    p = pd.read_csv(f"{base_dir}/round-1/prices_round_1_day_{d}.csv", sep=";")
    prices_df.append(p)

prices = pd.concat(prices_df, ignore_index=True)

# Continuous timestamp across days (1,000,000 ticks per day)
day_len = prices["timestamp"].max() + 100
prices["global_ts"] = (prices["day"] + 2) * day_len + prices["timestamp"]

prices.head()

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,global_ts
0,-2,0,AMETHYSTS,9996,1,9995.0,25.0,NaN,NaN,10004,1,10005.0,25.0,NaN,NaN,10000.0,0.0,0
1,-2,0,STARFRUIT,4997,26,NaN,NaN,NaN,NaN,5003,26,NaN,NaN,NaN,NaN,5000.0,0.0,0
2,-2,100,AMETHYSTS,10000,5,9995.0,20.0,NaN,NaN,10005,20,NaN,NaN,NaN,NaN,10002.5,0.0,100
3,-2,100,STARFRUIT,5000,5,4998.0,6.0,4997.0,20.0,5004,20,NaN,NaN,NaN,NaN,5002.0,0.0,100
4,-2,200,STARFRUIT,4997,1,4996.0,30.0,NaN,NaN,5003,31,NaN,NaN,NaN,NaN,5000.0,0.0,200


In [ ]:
# Load Trades
for d in days:
    t = pd.read_csv(f"{base_dir}/round-1/trades_round_1_day_{d}_nn.csv", sep=";")
    t.insert(0, "day", d)
    trades_df.append(t)

trades = pd.concat(trades_df, ignore_index=True)

# Filter out sample submission bot trades
trades = trades[(trades["buyer"] != "SUBMISSION") & (trades["seller"] != "SUBMISSION")].copy()

# Continuous timestamp across days (1,000,000 ticks per day)
day_len = prices["timestamp"].max() + 100
trades["global_ts"] = (trades["day"] + 2) * day_len + trades["timestamp"]

trades.head()

,day,timestamp,buyer,seller,symbol,currency,price,quantity,global_ts
0,-2,0,NaN,NaN,AMETHYSTS,SEASHELLS,9996.0,1,0
1,-2,0,NaN,NaN,STARFRUIT,SEASHELLS,5003.0,1,0
2,-2,0,NaN,NaN,AMETHYSTS,SEASHELLS,9996.0,1,0
3,-2,100,NaN,NaN,STARFRUIT,SEASHELLS,5004.0,6,100
4,-2,900,NaN,NaN,STARFRUIT,SEASHELLS,5001.0,2,900


In [10]:
# Order Book Imbalance and Spread
prices["spread"] = prices["ask_price_1"] - prices["bid_price_1"]

bid_vol_cols = ["bid_volume_1", "bid_volume_2", "bid_volume_3"]
ask_vol_cols = ["ask_volume_1", "ask_volume_2", "ask_volume_3"]

prices["total_bid_vol"] = prices[bid_vol_cols].fillna(0).sum(axis=1)
prices["total_ask_vol"] = prices[ask_vol_cols].fillna(0).sum(axis=1)

prices["obi_total"] = (prices["total_bid_vol"] - prices["total_ask_vol"]) / (prices["total_bid_vol"] + prices["total_ask_vol"])

prices.head()

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,...,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,global_ts,spread,total_bid_vol,total_ask_vol,obi_total
0,-2,0,AMETHYSTS,9996,1,9995.0,25.0,NaN,NaN,10004,...,25.0,NaN,NaN,10000.0,0.0,0,8,26.0,26.0,0.000000
1,-2,0,STARFRUIT,4997,26,NaN,NaN,NaN,NaN,5003,...,NaN,NaN,NaN,5000.0,0.0,0,6,26.0,26.0,0.000000
2,-2,100,AMETHYSTS,10000,5,9995.0,20.0,NaN,NaN,10005,...,NaN,NaN,NaN,10002.5,0.0,100,5,25.0,20.0,0.111111
3,-2,100,STARFRUIT,5000,5,4998.0,6.0,4997.0,20.0,5004,...,NaN,NaN,NaN,5002.0,0.0,100,4,31.0,20.0,0.215686
4,-2,200,STARFRUIT,4997,1,4996.0,30.0,NaN,NaN,5003,...,NaN,NaN,NaN,5000.0,0.0,200,6,31.0,31.0,0.000000


In [ ]:
# Join prevailing book snapshot onto each market trade
trades_enriched = trades.merge(prices, left_on=["day", "timestamp", "symbol"], right_on=["day", "timestamp", "product"], how="left")

# Classify aggressor side: +1 if lifting the ask, -1 if hitting the bid
trades_enriched["aggressor"] = np.where(trades_enriched["price"] >= trades_enriched["mid_price"], 1, -1)

trades_enriched.head()

,day,timestamp,buyer,seller,symbol,currency,price,quantity,global_ts_x,product,...,ask_price_3,ask_volume_3,mid_price,profit_and_loss,global_ts_y,spread,total_bid_vol,total_ask_vol,obi_total,aggressor
0,-2,0,NaN,NaN,AMETHYSTS,SEASHELLS,9996.0,1,0,AMETHYSTS,...,NaN,NaN,10000.0,0.0,0,8,26.0,26.0,0.000000,-1
1,-2,0,NaN,NaN,STARFRUIT,SEASHELLS,5003.0,1,0,STARFRUIT,...,NaN,NaN,5000.0,0.0,0,6,26.0,26.0,0.000000,1
2,-2,0,NaN,NaN,AMETHYSTS,SEASHELLS,9996.0,1,0,AMETHYSTS,...,NaN,NaN,10000.0,0.0,0,8,26.0,26.0,0.000000,-1
3,-2,100,NaN,NaN,STARFRUIT,SEASHELLS,5004.0,6,100,STARFRUIT,...,NaN,NaN,5002.0,0.0,100,4,31.0,20.0,0.215686,1
4,-2,900,NaN,NaN,STARFRUIT,SEASHELLS,5001.0,2,900,STARFRUIT,...,NaN,NaN,5002.5,0.0,900,3,32.0,30.0,0.032258,-1


In [11]:
# Sort chronologically per product
prices = prices.sort_values(["product", "global_ts"]).reset_index(drop=True)

prices.head()

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,...,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,global_ts,spread,total_bid_vol,total_ask_vol,obi_total
0,-2,0,AMETHYSTS,9996,1,9995.0,25.0,NaN,NaN,10004,...,25.0,NaN,NaN,10000.0,0.0,0,8,26.0,26.0,0.000000
1,-2,100,AMETHYSTS,10000,5,9995.0,20.0,NaN,NaN,10005,...,NaN,NaN,NaN,10002.5,0.0,100,5,25.0,20.0,0.111111
2,-2,200,AMETHYSTS,9996,1,9995.0,30.0,NaN,NaN,10004,...,30.0,NaN,NaN,10000.0,0.0,200,8,31.0,31.0,0.000000
3,-2,300,AMETHYSTS,9996,1,9995.0,26.0,NaN,NaN,10004,...,26.0,NaN,NaN,10000.0,0.0,300,8,27.0,27.0,0.000000
4,-2,400,AMETHYSTS,9995,20,NaN,NaN,NaN,NaN,9998,...,9.0,10005.0,20.0,9996.5,0.0,400,3,20.0,32.0,-0.230769


In [14]:
# 1. Compute forward price changes (in ticks/points) at multiple horizons
for k in [1, 5, 10, 20]:
    prices[f"fwd_ret_{k}"] = (prices.groupby(["day", "product"])["mid_price"].shift(-k) - prices["mid_price"])

prices.head()

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,...,profit_and_loss,global_ts,spread,total_bid_vol,total_ask_vol,obi_total,fwd_ret_1,fwd_ret_5,fwd_ret_10,fwd_ret_20
0,-2,0,AMETHYSTS,9996,1,9995.0,25.0,NaN,NaN,10004,...,0.0,0,8,26.0,26.0,0.000000,2.5,0.0,3.0,0.0
1,-2,100,AMETHYSTS,10000,5,9995.0,20.0,NaN,NaN,10005,...,0.0,100,5,25.0,20.0,0.111111,-2.5,-2.5,-1.5,-2.5
2,-2,200,AMETHYSTS,9996,1,9995.0,30.0,NaN,NaN,10004,...,0.0,200,8,31.0,31.0,0.000000,0.0,1.0,1.0,3.0
3,-2,300,AMETHYSTS,9996,1,9995.0,26.0,NaN,NaN,10004,...,0.0,300,8,27.0,27.0,0.000000,-3.5,0.0,2.5,0.0
4,-2,400,AMETHYSTS,9995,20,NaN,NaN,NaN,NaN,9998,...,0.0,400,3,20.0,32.0,-0.230769,3.5,3.5,0.0,3.5


In [16]:
# 2. Aggregate signed trade volume per tick and merge back onto prices
trades_enriched["signed_vol"] = trades_enriched["aggressor"] * trades_enriched["quantity"]
flow_per_tick = (trades_enriched.groupby(["day", "timestamp", "symbol"], as_index=False)["signed_vol"].sum().rename(columns={"symbol": "product"}))

trades_enriched.head()

,day,timestamp,buyer,seller,symbol,currency,price,quantity,global_ts_x,product,...,ask_volume_3,mid_price,profit_and_loss,global_ts_y,spread,total_bid_vol,total_ask_vol,obi_total,aggressor,signed_vol
0,-2,0,NaN,NaN,AMETHYSTS,SEASHELLS,9996.0,1,0,AMETHYSTS,...,NaN,10000.0,0.0,0,8,26.0,26.0,0.000000,-1,-1
1,-2,0,NaN,NaN,STARFRUIT,SEASHELLS,5003.0,1,0,STARFRUIT,...,NaN,5000.0,0.0,0,6,26.0,26.0,0.000000,1,1
2,-2,0,NaN,NaN,AMETHYSTS,SEASHELLS,9996.0,1,0,AMETHYSTS,...,NaN,10000.0,0.0,0,8,26.0,26.0,0.000000,-1,-1
3,-2,100,NaN,NaN,STARFRUIT,SEASHELLS,5004.0,6,100,STARFRUIT,...,NaN,5002.0,0.0,100,4,31.0,20.0,0.215686,1,6
4,-2,900,NaN,NaN,STARFRUIT,SEASHELLS,5001.0,2,900,STARFRUIT,...,NaN,5002.5,0.0,900,3,32.0,30.0,0.032258,-1,-2


In [ ]:
prices = prices.merge(flow_per_tick, on=["day", "timestamp", "product"], how="left")
prices["signed_vol"] = prices["signed_vol"].fillna(0)
prices["ofi_5"] = prices.groupby(["day", "product"])["signed_vol"].transform(lambda s: s.rolling(5, min_periods=1).sum())

prices.head()